# F1 — 금융 NER(8태그) 파인튜닝 · 팀원 분배 실행용 (Colab)

**무엇**: 후보 모델 하나를 골라 금융 라벨 28만 문장(GUIDE v1.2, Claude 라벨)으로 학습하고, 공통 규칙으로 채점한다. **4번 셀의 `RUN` 만 바꾸면 된다.** 누가 무엇을 돌릴지는 노션 분배표를 따른다.

## RUN 목록 (분배표와 같은 번호)
| RUN | 출발 모델 | 머리 | 입력 | 학습률 | A100 예상(추정) |
|---|---|---|---|---|---|
| 01 | 1단계 120만 roberta-base (사전학습 없음) | CRF | 원문 | 5e-5 | 1시간 안팎 |
| 02 | 경제 사전학습 roberta-base | CRF | 원문 | 5e-5 | 1시간 안팎 |
| 03 | 경제 사전학습 roberta-large | CRF | 원문 | 2e-5 | 2.5~3.5시간 |
| 03s | 경제 사전학습 roberta-large | softmax (CRF 효과 확인용) | 원문 | 2e-5 | 2~3시간 |
| 04 | 경제 사전학습 mmBERT 1에포크 | CRF | 형태소 | 5e-5 | 2~3시간 |
| 05 | 경제 사전학습 mmBERT 2에포크 | CRF | 형태소 | 5e-5 | 2~3시간 |
| 06 | 경제 사전학습 + 1단계 500만 roberta-base | CRF | 원문 | 5e-5 | 1시간 안팎 |
| 07 | 경제 사전학습 + 1단계 500만 roberta-large | CRF | 원문 | 2e-5 | 2.5~3.5시간 (모델 준비되면) |
| 08 | 경제 사전학습 + 1단계 500만 mmBERT | CRF | 형태소 | 5e-5 | 2~3시간 (모델 준비되면) |
| 10 | klue/bert-base (기준) | CRF | 원문 | 5e-5 | 1시간 안팎 |
| 11 | klue/roberta-base (기준, 사전학습 없음) | CRF | 원문 | 5e-5 | 1시간 안팎 |
| 12 | KoELECTRA-base-v3 (기준) | CRF | 원문 | 5e-5 | 1시간 안팎 |
| 13 | mmBERT-small (기준, 다국어) | CRF | 형태소 | 5e-5 | 1시간 안팎 |

공통: 3에폭, 배치 16, 최대 256토큰, 예열 10%, 감쇠 0.01, 시드 42. gpt-oss(#9)는 `E2_gptoss_금융NER_파인튜닝` 노트북.

## 채점 (모든 금융 노트북 공통, `fin_score.py`)
- 개체 = (시작·끝·태그) 글자 위치가 정답과 **정확히 같아야** 맞음. ALL = micro P·R·F1, MACRO_F1 = 고정 8태그 F1 평균.
- 채점 세트: **dev 1만**(에폭마다) → 끝나면 **test 1만**(한 번) · **test_2000**(gpt-oss와 같은 2천 문장, 공통 비교용) · (있으면) Fable 라벨 크롤링 문장.
- 오답 기록: `preds/*.jsonl` 에 문장마다 정답·예측·오답(경계 오류·태그 오류·없는 개체·놓침).
- 그림용 기록: 결과 json 에 에폭별 dev F1(`history`), 손실·학습률 전부(`log_history`), 태그별 점수.

## 런타임 · 저장
- **GPU**: A100(40GB 충분) 권장. A100이 없으면 base·small 모델(01·02·06·10~13)은 **L4**도 가능(약 2~3배 느림). large(03·03s·07)·mmBERT-base(04·05·08)는 A100. **T4 불가**(bf16 없음).
- **RAM**: 모두 표준(고용량 불필요).
- 모델·데이터를 Colab 로컬로 복사해 학습. 체크포인트는 2,000스텝·에폭 끝마다 로컬 + **드라이브 `results/_ckpt/`** → 끊기면 1~5번 다시 실행하면 이어서.
- 끝나면 6번 셀이 모델·결과·오답 파일을 드라이브 `results/{이름}/` 에 올리고, 7번 셀이 **런타임 반납(세션 종료)**.


In [6]:
# 1. GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다'
p = torch.cuda.get_device_properties(0); print(p.name, f'{p.total_memory/2**30:.0f}GB', 'bf16', torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), 'bf16 미지원 GPU(T4) → A100 으로 바꾸세요'

NVIDIA RTX PRO 6000 Blackwell Server Edition 95GB bf16 True


In [7]:
# 2. 드라이브 연결 · 폴더 찾기 (NER_라벨_통합_v1.2/finetune)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil, time
ROOT = '/content/drive/MyDrive/NER_라벨_통합_v1.2/finetune'   # 다르면 여기만 수정
if not os.path.exists(f'{ROOT}/fin_ft.py'):
    found = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/fin_ft.py', recursive=True)
             + glob.glob('/content/drive/Shareddrives/*/**/fin_ft.py', recursive=True)]
    print('찾은 위치:', found); assert found, 'fin_ft.py 를 못 찾음 → 공유 폴더를 내 드라이브에 바로가기로 추가했는지 확인'
    ROOT = found[0]
print('ROOT =', ROOT)
for f in ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'test_2000.jsonl']:
    print(f, sum(1 for _ in open(f'{ROOT}/data/{f}', encoding='utf-8')), '문장')
print('모델:', sorted(os.listdir(f'{ROOT}/models')))

Mounted at /content/drive
ROOT = /content/drive/MyDrive/NER_라벨_통합_v1.2/finetune
train.jsonl 279925 문장
dev.jsonl 10040 문장
test.jsonl 10035 문장
test_2000.jsonl 2000 문장
모델: ['01_stage1-1200k_roberta-base', '02_dapt_roberta-base', '03_dapt_roberta-large', '04_dapt_mmbert-ep1', '05_dapt_mmbert-ep2', '06_dapt-stage1-5000k_roberta-base', '07_dapt-stage1-5000k_roberta-large', '08_dapt-stage1-5000k_mmbert', '09_gptoss_dapt-lora']


In [8]:
# 3. 라이브러리
!pip install -q pytorch-crf kiwipiepy
import transformers; print('transformers', transformers.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
transformers 5.20.2


In [9]:
# 4. 실행할 RUN 고르기 (노션 분배표에서 내 번호)
RUN = '05'
CFG = {
 '01':  ('01_stage1-1200k_roberta-base',      'crf',     'none',  5e-5),
 '02':  ('02_dapt_roberta-base',              'crf',     'none',  5e-5),
 '03':  ('03_dapt_roberta-large',             'crf',     'none',  2e-5),
 '03s': ('03_dapt_roberta-large',             'softmax', 'none',  2e-5),
 '04':  ('04_dapt_mmbert-ep1',                'crf',     'morph', 5e-5),
 '05':  ('05_dapt_mmbert-ep2',                'crf',     'morph', 5e-5),
 '06':  ('06_dapt-stage1-5000k_roberta-base', 'crf',     'none',  5e-5),
 '07':  ('07_dapt-stage1-5000k_roberta-large','crf',     'none',  2e-5),
 '08':  ('08_dapt-stage1-5000k_mmbert',       'crf',     'morph', 5e-5),
 # 기준 모델(사전학습·1단계 없음): Hugging Face 에서 바로 받음
 '10':  ('hf:klue/bert-base',                         'crf', 'none',  5e-5),
 '11':  ('hf:klue/roberta-base',                      'crf', 'none',  5e-5),
 '12':  ('hf:monologg/koelectra-base-v3-discriminator','crf', 'none',  5e-5),
 '13':  ('hf:jhu-clsp/mmBERT-small',                  'crf', 'morph', 5e-5),
}
MODEL, HEAD, PRETOK, LR = CFG[RUN]
HF = MODEL.startswith('hf:')
EXP = f"F{RUN}_{MODEL[3:].replace('/', '-')}_{HEAD}"
L = '/content/local'; os.makedirs(L, exist_ok=True)
if HF: INIT = MODEL[3:]; copies = [(f'{ROOT}/data', f'{L}/data')]          # 기준 모델은 Hugging Face 에서 받음
else:
    assert os.path.exists(f'{ROOT}/models/{MODEL}/model.safetensors'), f'모델이 아직 없습니다: {MODEL}'
    INIT = f'{L}/model'; copies = [(f'{ROOT}/models/{MODEL}', f'{L}/model'), (f'{ROOT}/data', f'{L}/data')]
# 모델·데이터를 Colab 로컬로 복사(읽기 빠르게)
for src, dst in copies:
    t = time.time(); shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst); print('복사', dst, f'{time.time()-t:.0f}초')
EXTRA = f'{ROOT}/data/crawl350_fable.jsonl'   # Fable 라벨 크롤링 문장(준비되면 자동 채점)
print('RUN', RUN, '→', EXP, '| 머리', HEAD, '| 입력', PRETOK, '| lr', LR, '| 추가 채점', os.path.exists(EXTRA))

복사 /content/local/model 35초
복사 /content/local/data 2초
RUN 05 → F05_dapt_mmbert-ep2_crf | 머리 crf | 입력 morph | lr 5e-05 | 추가 채점 True


In [10]:
# 5. 학습 + 채점 (끊기면 1~5번 다시 실행 → 드라이브 체크포인트부터 이어서)
os.environ['TQDM_MININTERVAL'] = '60'   # 진행 막대를 60초에 한 번만 갱신 (출력 줄이기)
OUT = '/content/out'
EX = f'--eval_file "{EXTRA}"' if os.path.exists(EXTRA) else ''
!cd /content && python "{ROOT}/fin_ft.py" --init {INIT} --exp {EXP} --data {L}/data --head {HEAD} --pretok {PRETOK} --lr {LR} --out_root {OUT} --mirror "/content/ckpt_mirror" --epochs 3 --batch 16 --workers 8 --save_steps 6000 {EX}


10-27 11:12:32 학습 279,925 · dev 10,040 · test 10,035 · 태그 ['DATE', 'FIN_ORG', 'FIN_PRODUCT', 'INDICATOR', 'MONEY', 'ORG', 'POLICY', 'RATE'] · 머리 crf · 입력 morph
10-27 11:15:20 토큰화 166초 · 학습 문장당 토큰 53.7 · 잘린 문장 0
Loading weights: 100% 134/134 [00:00<00:00, 15261.13it/s]
[transformers] ModernBertModel LOAD REPORT from: /content/local/model
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
10-27 11:15:20 파라미터 306,953,044 · 에폭당 17,496스텝 · 총 52,488 · lr 5e-05 · 처음부터
{'loss': '165.5', 'grad_norm': '615.8', 'learning_rate': '9.432e-07', 'epoch': '0.005716'}
{'loss': '67.16', 'grad_norm': '114.5', 'learning_rate': '1.896e-06', 'epoch': '0.01143'}
{'loss': '27.31', 'grad_norm': '206.3', 'learning_rate': '2.849e-06', 'epoch': '0.01715'}
{'loss':

In [11]:
# 6. 결과 확인 → 모델·결과·오답 파일을 드라이브 results/EXP 에 올리기
r = json.load(open(f'{OUT}/results/{EXP}.json', encoding='utf-8'))
print('에폭별 dev:', r['history'])
for k in ['dev', 'test', 'test_2000', 'extra']:
    if k in r: print(f"{k:>9}: F1 {r[k]['ALL']['f1']} (P {r[k]['ALL']['precision']} R {r[k]['ALL']['recall']}) · macro {r[k]['MACRO_F1']} · 오답 {r[k]['error_types']}")
print('test 태그별 F1:', {t: r['test'][t]['f1'] for t in r['tags']})
print('학습', round(r['train_sec']/60, 1), '분 ·', r['gpu'])
UP = f'{ROOT}/results/{EXP}'; os.makedirs(UP, exist_ok=True)
shutil.copytree(f'{OUT}/models/{EXP}', f'{UP}/model', dirs_exist_ok=True)
shutil.copy(f'{OUT}/results/{EXP}.json', UP)
for f in glob.glob(f'{OUT}/preds/{EXP}_*.jsonl'): shutil.copy(f, UP)
same = all(os.path.getsize(f'{OUT}/models/{EXP}/{f}') == os.path.getsize(f'{UP}/model/{f}') for f in os.listdir(f'{OUT}/models/{EXP}'))
print('드라이브 업로드', UP, '· 크기 일치', same, '·', sorted(os.listdir(UP)))
OK = bool(r.get('done')) and same

에폭별 dev: [{'epoch': 1.0, 'step': 17496, 'dev_f1': 0.8989, 'dev_macro_f1': 0.8724}, {'epoch': 2.0, 'step': 34992, 'dev_f1': 0.9067, 'dev_macro_f1': 0.8844}, {'epoch': 3.0, 'step': 52488, 'dev_f1': 0.9113, 'dev_macro_f1': 0.8907}]
      dev: F1 0.9113 (P 0.9004 R 0.9225) · macro 0.8907 · 오답 {'spurious': 1063, 'boundary': 1047, 'missed': 1173, 'tag': 93, 'boundary+tag': 44}
     test: F1 0.9088 (P 0.8965 R 0.9214) · macro 0.8877 · 오답 {'spurious': 1120, 'boundary': 1119, 'missed': 1190, 'boundary+tag': 48, 'tag': 112}
test_2000: F1 0.9095 (P 0.8945 R 0.9249) · macro 0.8876 · 오답 {'boundary': 192, 'missed': 221, 'spurious': 246, 'boundary+tag': 11, 'tag': 29}
    extra: F1 0.8998 (P 0.8865 R 0.9136) · macro 0.8969 · 오답 {'missed': 40, 'boundary': 31, 'spurious': 36, 'tag': 2}
test 태그별 F1: {'DATE': 0.909, 'FIN_ORG': 0.9339, 'FIN_PRODUCT': 0.7797, 'INDICATOR': 0.8547, 'MONEY': 0.9945, 'ORG': 0.8989, 'POLICY': 0.7474, 'RATE': 0.9837}
학습 54.4 분 · NVIDIA RTX PRO 6000 Blackwell Server Edition
드라이브 

In [12]:
# 7. 런타임 반납(세션 종료) — 6번이 OK일 때만. 노션 분배표에 dev·test F1 을 적어 주세요
from google.colab import runtime
if OK: print('반납'); runtime.unassign()
else: print('업로드 확인 실패 → 반납하지 않음. 출력을 공유해 주세요')

반납
